<a href="https://colab.research.google.com/github/nexageapps/LLM/blob/main/01_Basic/L15_Project_Simple_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# L15: Project - Simple Chatbot

**Author:** Karthik Arjun  
**LinkedIn:** [Karthik Arjun](https://www.linkedin.com/in/karthik-arjun-a5b4a258/)  
**Level:** Basic  
**Lesson:** 15 of 15

---

## Learning Objectives

By the end of this lesson, you will:
- Build a functional chatbot that integrates L1–L14 concepts
- Implement conversation management and turn-taking
- Handle context (history) and keep it within model limits
- Generate responses using a language model
- Add simple memory (recent conversation history)
- Structure a minimal chat loop suitable for extension

---

## Concept: What We're Building

A **simple chatbot** that:
1. Keeps a **conversation history** (user and assistant messages)
2. **Formats context** for the model (e.g. "User: ...\nAssistant: ...")
3. **Trims or summarizes** context to fit the model's max length
4. **Generates** the next assistant reply given the context
5. **Updates memory** with the new turn

We use a small causal LM (e.g. GPT-2) and HuggingFace pipelines for generation.

---

In [ ]:
# Step 1: Install and Import

!pip install transformers torch accelerate -q

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Part 1: Conversation Memory

We store turns as a list of `{"role": "user"|"assistant", "content": "..."}` and cap the number of turns (or total tokens) to stay within the model's context window.

---

In [ ]:
# Step 2: Conversation History Class

class ConversationMemory:
    def __init__(self, max_turns=10, max_tokens=1024, tokenizer=None):
        self.max_turns = max_turns
        self.max_tokens = max_tokens
        self.tokenizer = tokenizer
        self.history = []

    def add_user(self, text):
        self.history.append({"role": "user", "content": text})

    def add_assistant(self, text):
        self.history.append({"role": "assistant", "content": text})

    def get_context_for_model(self):
        """Format recent history as a single string for the model."""
        lines = []
        for turn in self.history:
            prefix = "User:" if turn["role"] == "user" else "Assistant:"
            lines.append(f"{prefix} {turn['content']}")
        context = "\n".join(lines)
        if self.tokenizer and self.max_tokens:
            ids = self.tokenizer.encode(context)
            if len(ids) > self.max_tokens:
                ids = ids[-self.max_tokens:]
                context = self.tokenizer.decode(ids)
        return context

    def trim_to_max_turns(self):
        if len(self.history) > self.max_turns:
            self.history = self.history[-self.max_turns:]

    def clear(self):
        self.history = []

# Demo
mem = ConversationMemory(max_turns=6, max_tokens=256)
mem.add_user("Hello!")
mem.add_assistant("Hi! How can I help?")
mem.add_user("What's the weather like?")
print("Context string:")
print(mem.get_context_for_model())
print("\nMemory is ready for the model.")

## Part 2: Load Model and Generator

We use a text-generation pipeline with a small model (GPT-2) so it runs on CPU or a single GPU. You can swap in a larger model for better quality.

---

In [ ]:
# Step 3: Load Model and Pipeline

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

device = 0 if torch.cuda.is_available() else -1
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device
)

print(f"Model: {model_name}")
print(f"Max length (typical): 1024 tokens")

## Part 3: Response Generation

Given the conversation context, we append "Assistant:" and let the model generate the next reply. We stop at newlines or EOS to get one coherent turn.

---

In [ ]:
# Step 4: Generate One Assistant Reply

def generate_reply(generator, tokenizer, context, max_new_tokens=50, temperature=0.7):
    prompt = context + "\nAssistant:"
    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    full = out[0]["generated_text"]
    # Take only the new part after the prompt
    new_part = full[len(prompt):].strip()
    # Stop at first newline for a single reply
    if "\n" in new_part:
        new_part = new_part.split("\n")[0].strip()
    return new_part if new_part else "(no response)"

# Quick test with minimal context
test_ctx = "User: What is 2+2?\nAssistant:"
reply = generate_reply(generator, tokenizer, test_ctx, max_new_tokens=30)
print("Test prompt:", repr(test_ctx))
print("Generated reply:", repr(reply))

## Part 4: Chat Loop

Tie memory, context, and generation together: for each user message we update memory, build context, generate a reply, then add it to memory and trim if needed.

---

In [ ]:
# Step 5: Simple Chatbot Class

class SimpleChatbot:
    def __init__(self, generator, tokenizer, max_turns=10, max_context_tokens=512, max_new_tokens=60, temperature=0.7):
        self.generator = generator
        self.tokenizer = tokenizer
        self.memory = ConversationMemory(
            max_turns=max_turns,
            max_tokens=max_context_tokens,
            tokenizer=tokenizer
        )
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature

    def chat(self, user_message):
        self.memory.add_user(user_message)
        context = self.memory.get_context_for_model()
        reply = generate_reply(
            self.generator,
            self.tokenizer,
            context,
            max_new_tokens=self.max_new_tokens,
            temperature=self.temperature
        )
        self.memory.add_assistant(reply)
        self.memory.trim_to_max_turns()
        return reply

    def clear_history(self):
        self.memory.clear()

# Run a short conversation
bot = SimpleChatbot(generator, tokenizer, max_turns=6, max_context_tokens=256, max_new_tokens=40)

print("=== Simple Chatbot Demo ===\n")
for user_msg in ["Hi!", "What can you do?", "Tell me a one-sentence fact about Python."]:
    print(f"User: {user_msg}")
    reply = bot.chat(user_msg)
    print(f"Assistant: {reply}\n")

## Part 5: Interactive Chat (Optional)

In a notebook you can run a loop that asks for user input and prints the assistant reply. In Colab/terminal you can type in the box or use a simple `input()` loop.

---

In [ ]:
# Step 6: Interactive Loop (run and type in the box; type 'quit' to stop)

def run_chat(bot, quit_word="quit"):
    print("Chat with the bot. Type", repr(quit_word), "to end.\n")
    while True:
        user_input = input("You: ").strip()
        if not user_input:
            continue
        if user_input.lower() == quit_word:
            print("Goodbye!")
            break
        reply = bot.chat(user_input)
        print(f"Bot: {reply}\n")

# Uncomment to run interactively:
# run_chat(bot)
print("To chat interactively, create a SimpleChatbot and call run_chat(bot).")

## Project Requirements and Extensions

Your **functional chatbot** should:
- [ ] Maintain conversation history (memory)
- [ ] Use context (recent turns) when generating
- [ ] Handle token limits (trim or truncate context)
- [ ] Generate one reply per user message
- [ ] Support clearing history and optional system prompt

**Ideas to extend:**
- Add a system prompt (e.g. "You are a helpful assistant.") at the start of context.
- Try different models (e.g. `gpt2-medium`, or an instruction-tuned model) and compare quality.
- Add simple intent detection or keyword triggers (e.g. "weather", "joke") and custom responses.
- Log conversations to a file for debugging or training data.
- Implement a simple RAG: retrieve a few relevant sentences from a doc and prepend them to context.

---

## Exercises

### Exercise 1: System Prompt
Add an optional `system_prompt` to `SimpleChatbot` that is always prepended to the context (e.g. "You are a friendly coding assistant."). Compare replies with and without it.

### Exercise 2: Context Length
Vary `max_turns` and `max_context_tokens`. For 3 fixed user questions, report how reply quality or relevance changes as you increase or decrease context.

### Exercise 3: Memory Beyond Turns
Implement a "summary" of older turns: when trimming, replace the oldest turns with one summarized sentence (e.g. using the same model or a rule-based shortening) so the bot keeps a high-level memory of the full conversation.

### Exercise 4: Evaluation
Using L14 metrics: collect 5 user inputs and ideal reference replies. Run your chatbot on the same inputs and compute BLEU/ROUGE (or a simple overlap metric) between bot output and reference.

### Exercise 5: Minimal RAG
Maintain a small list of facts (e.g. 5 sentences). For each user message, pick the most relevant fact (e.g. by keyword overlap) and prepend it to the context as "Relevant info: ...". Observe whether answers improve for fact-based questions.

---

## Key Takeaways

1. A **simple chatbot** = conversation memory + context formatting + LM generation.
2. **Context** must be trimmed (turns or tokens) to fit the model.
3. **Memory** can be full history, sliding window, or summarized history.
4. You can extend with system prompts, RAG, or better models.
5. Use L14 metrics to evaluate reply quality against references.

---

## Additional Resources

- HuggingFace [Conversational Pipeline](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ConversationalPipeline)
- [Prompt Engineering](https://www.promptingguide.ai/) for better system/user formatting
- L10 (generation), L11 (prompts), L14 (evaluation)

---

## Next Steps

You have completed **Level 1 (Basic)**. Next: **Level 2 (Intermediate)** – Transfer learning, fine-tuning, LoRA, BERT/GPT/T5, RAG, and the Document QA project (L16–L30).

---

**Congratulations! You built a simple chatbot with memory and context.**